# 02 - MCP for AI agents

## What is MCP?

**MCP (Model Context Protocol)** is one standard interface for agents to talk to any external tool or data source.

- **Without MCP:** every tool needs custom glue code. A different integration for Google, another for the filesystem, another for Wikipedia. Each one breaks differently.
- **With MCP:** every tool speaks the same language. Any MCP-compatible server just works.

## Three components

- **Host** is the application. It owns the agent and starts everything.
  - Host = the app that runs the agent and manages everything. It creates the clients.
- **Client** lives inside the host, one per server. Talks to its server via JSON-RPC 2.0 messages.
  - Client = "please do this" (asks). Lives inside the host.
- **Server** exposes one capability. Receives the message, runs the function, returns the result.
  - Server = "done, here's the result" (does). Lives in the tool's world.

MCP is what lets them talk across that boundary.

## JSON-RPC 2.0 message workflow

Each request is a JSON object with a method, params, and an id. The response matches the same id back.

![JSON-RPC 2.0 message workflow between agent, client, and server](diagrams/json_rpc_workflow.svg)

## Example: research agent

Goal: *"Research transformer models and save a summary to my desktop."*

Each server exposes different primitives, 1) tools, 2) resources, or 3) prompts, depending on what it does.

What each server exposes:
- **Tools** are callable functions the agent invokes to take action. e.g. `search_web`, `save_file`
- **Resources** are read-only documents the agent reads as context. e.g. the Wikipedia article
- **Prompts** are reusable persona templates the server provides so the agent doesn't hardcode them

In short: Tools = do something. Resources = read something. Prompts = shape the conversation.

![Research agent: the host holds the agent and three clients, each talking JSON-RPC to its own server](diagrams/research_agent.svg)

## Deployment patterns

Where an MCP server runs depends on what it does and who needs it.

- **Local deployment** runs the server on the same machine as the agent. Simple, no network needed, good for personal tools like filesystem access. This is what the research agent example assumes.
- **Remote deployment** runs the server in the cloud or on another machine. The agent talks to it over HTTP. Used when the tool is a shared service or too heavy to run locally.
- **Hybrid architecture** mixes both. Some servers run locally, others remotely. Most real-world agents end up here.
  - e.g. filesystem server is local, web search API server is remote

## Setup

In [1]:
import os

from agents import Agent, ModelSettings, Runner, trace
from dotenv import load_dotenv

load_dotenv('C:/Users/kongl/Starting-something-new/.env')
print("API key found:", bool(os.environ.get("OPENAI_API_KEY")))

API key found: True


## Step 1: Build the server

So far, everything has been conceptual. Now it's time to build it.

The MCP server is just a Python script. It doesn't know anything about agents or LLMs - it just sits there, waiting to be called. When an agent needs something, it sends a JSON-RPC message, and the server handles it.

The first server is intentionally simple: one tool, one resource, one prompt. Think of it as the "Hello World" of MCP - just enough to see all three primitives working together before things get more complex.

The server lives in `mcp_demo_server.py` alongside this notebook. It has to be a standalone file because the agent launches it as a subprocess.

In [2]:
# server.py
from mcp.server.mcpserver import MCPServer

# Initialize the server with a name
mcp = MCPServer("DemoServer")

def log(output: str, file_name: str = "server_output.log"):
    """Log output to a file"""
    with open(file_name, "a") as log_file:
        log_file.write(output + "\n")

# Resource: a personalized greeting the agent can read
@mcp.resource("greeting://{name}")
def get_greeting(name: str) -> str:
    """Return a personalized greeting for the given name"""
    log(f"Received request for greeting: {name}")
    return f"Greetings, {name}! (from resource)"

# Tool: a function the agent can call to do something
@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two integers and return the sum"""
    log(f"Adding {a} and {b}")
    return a + b

# Prompt: a reusable template the server provides
@mcp.prompt()
def welcome(name: str) -> str:
    """Generate a welcome message using the greeting resource"""
    greeting_text = get_greeting(name)
    log(f"Creating welcome message for: {name}")
    return f"{greeting_text} How can I assist you today?"

What each part does:

- `MCPServer("DemoServer")` creates the server. It handles all the protocol wiring so the focus can stay on what to expose.
- `log()` is a debugging helper. It writes to a file so it's possible to see what the server received and when.
- `@mcp.resource("greeting://{name}")` registers a read-only resource at a URI. The agent fetches it like a document, no action happens.
- `@mcp.tool()` registers a callable function. The agent invokes `add` when it needs to compute something. The server runs it and returns the result.
- `@mcp.prompt()` registers a reusable template. The host can request it instead of hardcoding the prompt in the agent.

Nothing runs yet. The server is just registered and waiting. Step 2 is the agent that connects to it.

## Step 2: Connect the agent

The agent launches the server as a subprocess over stdio, then sends it messages. This is the local deployment pattern - both run on the same machine, no network needed.

`MCPServerStdio` is the client. It starts the server process, keeps the connection open for the duration of the `async with` block, and shuts it down cleanly when done.

In [3]:
import os
import sys
from pathlib import Path
from agents import Agent, Runner
from agents.mcp import MCPServerStdio, MCPServerStdioParams
from dotenv import load_dotenv
from mcp.client.stdio import stdio_client

load_dotenv('C:/Users/kongl/Starting-something-new/.env')

SCRIPT = Path("mcp_demo_server.py").resolve()      # server file next to this notebook
MCP_EXE = Path(sys.executable).parent / "mcp.exe" # mcp CLI from our venv

# Notebook-only fix: the server's error output goes to Jupyter's stderr by default,
# which Windows can't hand to a subprocess. Send it to a real (empty) file instead.
class NotebookMCPServerStdio(MCPServerStdio):
    def create_streams(self):
        return stdio_client(self.params, errlog=open(os.devnull, "w"))

async def main():
    async with NotebookMCPServerStdio(
        name="MCP Server",
        params=MCPServerStdioParams(
            command=str(MCP_EXE),
            args=["run", str(SCRIPT)],
        ),
    ) as mcp_server:                           # launches server subprocess, keeps connection open
        agent = Agent(
            name="Assistant",
            instructions=(
                "You are a helpful assistant. "
                "For math calculations, always use the add tool. "
                "For greeting the user by name, use the greeting resource or prompt from the mcp server."
            ),
            mcp_servers=[mcp_server],          # agent knows which servers it can talk to
        )
        result = await Runner.run(agent, "Hello, my name is Alice. What is 5 + 7?")
        # agent reasons -> calls add(5,7) via JSON-RPC -> server returns 12 -> agent replies
        print(result.final_output)

# What happens when this runs:
# 1. The agent launches mcp_demo_server.py as a subprocess
# 2. The agent receives: 'Hello, my name is Alice. What is 5 + 7?'
# 3. The agent reasons: 'I need to add numbers, there is a tool for that'
# 4. It sends a JSON-RPC request to the server: add(5, 7)
# 5. The server runs add(), returns 12
# 6. The agent uses the result to reply: 'Hello Alice! 5 + 7 = 12.'
# Note: the LLM never computed 5 + 7 itself, it just decided what to call

await main()

[11/11/26 23:13:44] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582568;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582569;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

[11/11/26 23:13:45] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582574;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582575;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

Hello Alice! 5 + 7 is 12.


## Step 3: Give the Research Planner's tool its own server

My Research Planner already had a tool, `get_research_sources()`. It was written right inside the agent code, so only that one agent could use it.

Here the same function moves into its own MCP server, `mcp_research_server.py`. The agent no longer owns the tool. It finds it by connecting to the server.

- **Before:** the tool lived in the agent code, passed with `tools=[...]`
- **After:** the tool lives in a server, passed with `mcp_servers=[...]`

Now any MCP-compatible host can use it, another agent or Claude Desktop included.

In [4]:
# mcp_research_server.py
from mcp.server.mcpserver import MCPServer

mcp = MCPServer("Research Tools")

@mcp.tool()                                # same function as the Research Planner, new home
def get_research_sources() -> list[str]:
    """Provides a list of research sources."""
    return ["Wikipedia", "Google", "YouTube"]

In [5]:
RESEARCH_SCRIPT = Path("mcp_research_server.py").resolve()

async def research():
    async with NotebookMCPServerStdio(         # same notebook-safe client from Step 2
        name="Research Tools",
        params=MCPServerStdioParams(
            command=str(MCP_EXE),
            args=["run", str(RESEARCH_SCRIPT)],
        ),
    ) as research_server:
        agent = Agent(
            name="Assistant",
            instructions="Use the research tools to perform research.",
            mcp_servers=[research_server],     # no tools=[...] here, the server provides them
        )
        result = await Runner.run(agent, "Get the available research sources")
        print(result.final_output)

# What happens when this runs:
# 1. The agent launches mcp_research_server.py as a subprocess
# 2. It asks the server which tools exist and finds get_research_sources
# 3. The agent receives: 'Get the available research sources'
# 4. It sends a JSON-RPC request to the server: get_research_sources()
# 5. The server returns ['Wikipedia', 'Google', 'YouTube']
# 6. The agent turns that list into its reply

await research()

[11/11/26 23:13:47] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582580;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582581;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

[11/11/26 23:13:48] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582586;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582587;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

Available research sources:
- Wikipedia
- Google
- YouTube


**How the connection works**

- `async def research():` only defines the function. Nothing runs yet.
- `await research()` runs it and waits until it finishes. Most of the time is spent waiting on the server and on OpenAI, and `async` handles that waiting. Jupyter allows `await` directly in a cell. A plain `.py` script uses `asyncio.run(research())` instead.

The `async with` block is like a phone call:

- **Enter (dial):** the server starts as a separate process and connects. The agent asks it which tools exist and finds `get_research_sources`.
- **Inside (talk):** the connection stays open, so the agent can call the server as many times as it needs.
- **Exit (hang up):** when the indented block ends, the connection closes and the server process stops. This happens even if an error occurs, so no stray server is left running.

Every run of the cell starts a fresh call with a fresh server. That's why the agent is created and run inside the block. Outside it, the server is gone.

## Step 4: Run the server on its own (remote pattern)

So far the agent started the server itself and shut it down when done. That's the local pattern: the server only lives while the agent needs it.

Now the same research server runs on its own as a small web server at `http://localhost:8000/sse`. The agent doesn't start anything. It just connects over HTTP, the same way it would reach a server in the cloud.

- **Local (stdio):** agent starts the server, talks through stdin/stdout, server stops with the agent
- **Remote (SSE):** server is already running, agent connects by URL, server keeps running after the agent leaves

The server file doesn't change at all. Only the transport changes, from `stdio` to `sse`.

In [6]:
import socket
import subprocess
import time

# Normally this server runs in its own terminal or on another machine:
#   mcp run mcp_research_server.py -t sse
# Here the notebook starts it in the background so Run All still works.
server_process = subprocess.Popen(
    [str(MCP_EXE), "run", str(RESEARCH_SCRIPT), "-t", "sse"],  # -t sse = serve over HTTP
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Wait until the server is accepting connections on port 8000
for _ in range(50):
    try:
        socket.create_connection(("127.0.0.1", 8000), timeout=0.2).close()
        break
    except OSError:
        time.sleep(0.2)

print("Server running at http://localhost:8000/sse")

                    INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=14582593;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582594;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1084\1084]8;;\
                             204 No Content"                                                                       

                    INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=14582599;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582600;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1084\1084]8;;\
                             204 No Content"                                                                       

Server running at http://localhost:8000/sse


In [7]:
from agents.mcp import MCPServerSse

async def research_remote():
    async with MCPServerSse(                       # client that connects by URL
        name="SSE Python Server",
        params={"url": "http://localhost:8000/sse"},  # no command, no script, just an address
    ) as research_server:
        agent = Agent(
            name="Assistant",
            instructions="Use the research tools to perform research.",
            mcp_servers=[research_server],
        )
        result = await Runner.run(agent, "Get the available research sources")
        print(result.final_output)

# What happens when this runs:
# 1. The agent connects to the server's URL (the server is already running)
# 2. It opens one long-lived GET /sse connection, the channel for the server's replies
# 3. Each message the agent sends is a POST: handshake, list tools, call get_research_sources()
# 4. The server returns ['Wikipedia', 'Google', 'YouTube'] over the /sse channel
# 5. The agent turns that list into its reply
# 6. Leaving the block hangs up, but the server keeps running

await research_remote()

[11/11/26 23:13:49] INFO     HTTP Request: GET http://localhost:8000/sse "HTTP/1.1 200 OK"          ]8;id=14582605;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582606;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\

                    INFO     HTTP Request: POST                                                     ]8;id=14582611;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582612;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             http://localhost:8000/messages/?session_id=f54066f65b774afca0e6201157c                
                             12479 "HTTP/1.1 202 Accepted"                                                         

                    INFO     HTTP Request: POST                                                     ]8;id=14582617;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582618;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             http://localhost:8000/messages/?session_id=f54066f65b774afca0e6201157c                
                             12479 "HTTP/1.1 202 Accepted"                                                         

[11/11/26 23:13:50] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582623;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582624;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

                    INFO     HTTP Request: POST                                                     ]8;id=14582629;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582630;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             http://localhost:8000/messages/?session_id=f54066f65b774afca0e6201157c                
                             12479 "HTTP/1.1 202 Accepted"                                                         

                    INFO     HTTP Request: POST                                                     ]8;id=14582635;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582636;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             http://localhost:8000/messages/?session_id=f54066f65b774afca0e6201157c                
                             12479 "HTTP/1.1 202 Accepted"                                                         

[11/11/26 23:13:51] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582641;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582642;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

The available research sources are:

- Wikipedia  
- Google  
- YouTube


In [8]:
# The agent is done, but the server is still running and could serve other agents.
# Stopping it is a separate step, the opposite of Step 3.
server_process.terminate()
server_process.wait()
print("Server stopped")

Server stopped


## Step 5: Plug in a ready-made server

Every server so far was one I wrote. This time the agent uses a server someone else built: the official MCP filesystem server. It's written in JavaScript, not Python, and runs with `npx`.

This is the promise from the start of the notebook. The agent code barely changes, yet it suddenly knows how to list, read, and search files. No glue code, no custom integration.

- **My servers:** started with `mcp run my_file.py`
- **Ready-made server:** started with `npx -y @modelcontextprotocol/server-filesystem@2.2.0 <folder>`

**Safety note:** the server can only touch the folder passed to it, here the notebook's own folder. Inside that folder it has write tools too (`write_file`, `edit_file`, `move_file`), so the agent could change files if asked.

In [9]:
import shutil

# Node.js was installed while VS Code was open, so add it to PATH if this kernel can't see it yet
if shutil.which("npx") is None:
    os.environ["PATH"] += os.pathsep + r"C:\Program Files\nodejs"

NOTEBOOK_FOLDER = str(Path.cwd())  # the only folder the server is allowed to access

async def browse_files():
    async with NotebookMCPServerStdio(             # same notebook-safe client from Step 2
        name="Filesystem Server, via npx",
        params={
            "command": "npx",                    # Node's package runner, not mcp run
            "args": ["-y", "@modelcontextprotocol/server-filesystem@2.2.0", NOTEBOOK_FOLDER],
        },
        client_session_timeout_seconds=60,          # first run downloads the package, so allow time
    ) as fs_server:
        tools = await fs_server.list_tools()        # see what this ready-made server offers
        print(f"Server offers {len(tools)} tools:", ", ".join(t.name for t in tools), "\n")

        agent = Agent(
            name="Filesystem Agent",
            instructions="Use the filesystem tools to help the user with their tasks.",
            mcp_servers=[fs_server],
        )
        result = await Runner.run(agent, "List all file names in the current directory")
        print(result.final_output)

# What happens when this runs:
# 1. npx downloads the filesystem server (first run only) and starts it
# 2. The agent asks the server which tools exist and gets back a whole toolkit
# 3. The agent receives: 'List all file names in the current directory'
# 4. It picks list_directory and sends a JSON-RPC request for the notebook folder
# 5. The server reads the folder and returns the names
# 6. The agent formats them into its reply
# Note: none of these file tools were written by me, the server brought them

await browse_files()

Server offers 16 tools: read_file, read_text_file, read_media_file, read_multiple_files, write_file, append_file, edit_file, create_directory, list_directory, list_directory_with_sizes, directory_tree, move_file, copy_file, search_files, get_file_info, list_allowed_directories 



[11/11/26 23:13:53] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582647;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582648;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

                    INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=14582653;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582654;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1084\1084]8;;\
                             204 No Content"                                                                       

[11/11/26 23:13:54] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582659;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582660;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

Here are the entries in the current directory:

- 02_MCP_for_ai_agents.ipynb
- diagrams/
- experiments/
- mcp_demo_server.py
- mcp_research_server.py
- mcp_time_travel_tracker.py
- server_output.log
- tracker_server.log
- _check_outputs.py
- __pycache__/


## Step 6: A time-travel journal (before MCP)

A small project: an agent that keeps a journal of time-travel trips. It has two tools:

- `record_event` saves a new trip to the journal
- `load_journal` reads back every trip so far

The key idea: every `Runner.run` is a brand-new conversation, so the agent remembers nothing between trips on its own. The journal is its memory. The agent reads it, adds to it, and reads it again.

For now the journal is a plain Python list and the tools are `@function_tool`s inside the agent, the same way the Research Planner's tool started. The next step moves this journal into an MCP server.

In [10]:
from agents import function_tool

_journal = []  # the agent's memory, a plain list living in this notebook (reset each time this cell runs)

@function_tool
def record_event(entry: str) -> dict:
    """Add a new travel event to the journal."""
    _journal.append(entry)
    print(f"Event recorded: {entry}")             # shows each time the agent calls this tool
    return {"status": "recorded", "entry": entry}

@function_tool
def load_journal() -> dict:
    """Load the current travel journal entries."""
    print("Loading journal entries...")
    return {"status": "loaded", "journal": "\n".join(_journal)}

journal_agent = Agent(
    name="Time Tracker Agent",
    instructions="""You are a time tracking journaling agent.
Always use the 'load_journal' tool at the start to get past entries.
For a new event, call 'record_event' to save it.
If asked for a summary or to show the journal, output all recorded events.""",
    tools=[record_event, load_journal],            # tools live inside the agent, no server
)

travel_events = [
    "Traveled to Ancient Rome and watched a gladiator fight",
    "Visited the signing of the Declaration of Independence in 1776",
    "Witnessed the moon landing in 1969",
]

async def time_travel():
    print("Recording travels:")
    for event in travel_events:
        await Runner.run(journal_agent, event)   # a separate conversation per trip
    result = await Runner.run(journal_agent, "Show my travel history")
    print("\nFinal Journal:")
    print(result.final_output)

# What happens when this runs:
# 1. Each trip is sent to the agent as its own fresh conversation
# 2. The agent calls load_journal first, then record_event to save the trip
# 3. After three trips, the list holds all three entries
# 4. 'Show my travel history' is yet another fresh conversation
# 5. The agent calls load_journal and gets all three trips back
# 6. It writes them up as the final journal
# Note: the agent only 'remembers' the trips because the journal tool does

await time_travel()

Recording travels:


[11/11/26 23:13:55] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582665;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582666;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

Loading journal entries...


[11/11/26 23:13:56] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582671;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582672;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

Event recorded: Traveled to Ancient Rome and watched a gladiator fight


[11/11/26 23:13:57] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582677;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582678;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

... (12 outputs left out)

                    INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582731;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582732;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

Loading journal entries...


[11/11/26 23:14:06] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582737;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582738;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   


Final Journal:
Here’s your recorded travel history:

1. Traveled to Ancient Rome and watched a gladiator fight  
2. Visited the signing of the Declaration of Independence in 1776  
3. Witnessed the moon landing in 1969


## Step 7: Move the journal into an MCP server

Same journal, same two tools, new home. They now live in `mcp_time_travel_tracker.py`, so any agent can share the journal instead of one agent owning it.

- **Step 6:** journal and tools inside the agent, passed with `tools=[...]`
- **Step 7:** journal and tools inside a server, passed with `mcp_servers=[...]`

The server file (shown here, not run in the notebook):

```python
import sys
from mcp.server.mcpserver import MCPServer

mcp = MCPServer("Time Travel Tracker")
_journal = []  # lives as long as the server process lives

def log(message):
    print(message, file=sys.stderr, flush=True)  # stderr, not stdout

@mcp.tool()
def record_event(entry: str) -> dict:
    """Add a new travel event to the journal."""
    _journal.append(entry)
    log(f"Event recorded: {entry}")
    return {"status": "recorded", "entry": entry}

@mcp.tool()
def load_journal() -> dict:
    """Load the current travel journal entries."""
    log("Loading journal entries...")
    return {"status": "loaded", "journal": "\n".join(_journal)}

if __name__ == "__main__":
    mcp.run(transport="sse")  # running the file directly serves over HTTP
```

**Lesson learned:** the first version used plain `print()` in the tools and broke the connection with `Failed to parse JSONRPC message`. Over stdio, the server's stdout *is* the JSON-RPC channel, so any stray print lands in the middle of the protocol. Logs go to stderr instead.

In [11]:
TRACKER_SCRIPT = Path("mcp_time_travel_tracker.py").resolve()

JOURNAL_INSTRUCTIONS = """You are a time-travel journaling agent.
Always use the 'load_journal' tool at the start to get past entries.
For a new event, call 'record_event' to save it.
If asked for a summary or to show the journal, output all recorded events."""

async def time_travel_mcp():
    async with NotebookMCPServerStdio(             # same notebook-safe client from Step 2
        name="Time Tracker Server",
        params=MCPServerStdioParams(
            command=str(MCP_EXE),
            args=["run", str(TRACKER_SCRIPT)],
        ),
    ) as tracker_server:
        agent = Agent(
            name="Assistant",
            instructions=JOURNAL_INSTRUCTIONS,
            mcp_servers=[tracker_server],          # tools come from the server now
        )
        print("Recording travels:")
        for event in travel_events:                # same three trips as Step 6
            await Runner.run(agent, event)
        result = await Runner.run(agent, "Show my travel history")
        print("\nFinal Journal:")
        print(result.final_output)

# What happens when this runs:
# 1. The agent starts the journal server and finds record_event and load_journal
# 2. Each trip is a fresh conversation, but all of them talk to the same running server
# 3. The server's _journal list fills up with the three trips
# 4. 'Show my travel history' calls load_journal and gets all three back
# 5. Leaving the block stops the server, and the journal goes with it

await time_travel_mcp()

Recording travels:


[11/11/26 23:14:07] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582743;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582744;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

[11/11/26 23:14:08] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582749;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582750;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

[11/11/26 23:14:09] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=14582755;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582756;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1084\1084]8;;\
                             204 No Content"                                                                       

                    INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582761;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582762;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

[11/11/26 23:14:10] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582767;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582768;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

... (5 outputs left out)

                    INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582803;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582804;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

[11/11/26 23:14:16] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582809;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582810;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

[11/11/26 23:14:17] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582815;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582816;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   


Final Journal:
Here’s your recorded time-travel history so far:

1. Traveled to Ancient Rome and watched a gladiator fight  
2. Visited the signing of the Declaration of Independence in 1776  
3. Witnessed the moon landing in 1969


## Step 8: A journal that outlives the agent

In Step 7 the journal disappeared the moment the agent hung up, because the server stopped with it.

Running the same server over SSE changes that. The server keeps running on its own, so its memory does too. To prove it, two completely separate sessions connect:

- **Session 1** records the trips, then hangs up
- **Session 2** is a brand-new connection and only asks for the history

If session 2 sees the trips, the memory lives in the server, not the agent.

In [12]:
# Start the journal server on its own. Running the file directly uses its
# __main__ block: mcp.run(transport="sse"). Its logs go to a file to read later.
tracker_log = open("tracker_server.log", "w")
tracker_process = subprocess.Popen(
    [sys.executable, str(TRACKER_SCRIPT)],
    stdout=subprocess.DEVNULL,
    stderr=tracker_log,
)

for _ in range(50):                                # wait for port 8000 to open
    try:
        socket.create_connection(("127.0.0.1", 8000), timeout=0.2).close()
        break
    except OSError:
        time.sleep(0.2)

async def journal_session(prompts):
    """One agent session: connect, run the prompts, hang up."""
    async with MCPServerSse(
        name="Time Tracker Server",
        params={"url": "http://localhost:8000/sse"},
    ) as tracker_server:
        agent = Agent(name="Assistant", instructions=JOURNAL_INSTRUCTIONS, mcp_servers=[tracker_server])
        for prompt in prompts:
            result = await Runner.run(agent, prompt)
        return result.final_output

await journal_session(travel_events)               # session 1: record the trips
print("Session 1 recorded the trips and hung up.\n")

history = await journal_session(["Show my travel history"])  # session 2: brand-new connection
print("Session 2 asked for the history:")
print(history)

[11/11/26 23:14:18] INFO     HTTP Request: GET http://localhost:8000/sse "HTTP/1.1 200 OK"          ]8;id=14582821;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582822;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\

[11/11/26 23:14:19] INFO     HTTP Request: POST                                                     ]8;id=14582827;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582828;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             http://localhost:8000/messages/?session_id=1f0a3cdb14394977ad4dca588b7                
                             f6d12 "HTTP/1.1 202 Accepted"                                                         

                    INFO     HTTP Request: POST                                                     ]8;id=14582833;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582834;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             http://localhost:8000/messages/?session_id=1f0a3cdb14394977ad4dca588b7                
                             f6d12 "HTTP/1.1 202 Accepted"                                                         

[11/11/26 23:14:20] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14582839;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582840;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

                    INFO     HTTP Request: POST                                                     ]8;id=14582845;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582846;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             http://localhost:8000/messages/?session_id=1f0a3cdb14394977ad4dca588b7                
                             f6d12 "HTTP/1.1 202 Accepted"                                                         

                    INFO     HTTP Request: POST                                                     ]8;id=14582851;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14582852;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             http://localhost:8000/messages/?session_id=1f0a3cdb14394977ad4dca588b7                
                             f6d12 "HTTP/1.1 202 Accepted"                                                         

... (28 outputs left out)

                    INFO     HTTP Request: POST                                                     ]8;id=14583019;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583020;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             http://localhost:8000/messages/?session_id=41c28f9a41f246dca8f734cc179                
                             ebca8 "HTTP/1.1 202 Accepted"                                                         

                    INFO     HTTP Request: POST                                                     ]8;id=14583025;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583026;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             http://localhost:8000/messages/?session_id=41c28f9a41f246dca8f734cc179                
                             ebca8 "HTTP/1.1 202 Accepted"                                                         

[11/11/26 23:14:31] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14583031;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583032;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

Session 2 asked for the history:
Here’s your recorded time‑travel history:

1. Traveled to Ancient Rome and watched a gladiator fight  
2. Visited the signing of the Declaration of Independence in 1776  
3. Witnessed the moon landing in 1969


In [13]:
# Stop the server. Only now does the journal disappear.
tracker_process.terminate()
tracker_process.wait()
tracker_log.close()

# What the server did while the two sessions came and went:
print("Server log:")
with open("tracker_server.log") as f:
    print("".join(line for line in f if "Event recorded" in line or "Loading" in line))

Server log:
Loading journal entries...
Event recorded: Traveled to Ancient Rome and watched a gladiator fight
Loading journal entries...
Event recorded: Visited the signing of the Declaration of Independence in 1776
Loading journal entries...
Event recorded: Witnessed the moon landing in 1969
Loading journal entries...



# Experiments

Five experiments to push on what the steps above showed. Each one starts with a question. The servers they use live in the `experiments` folder.

## Experiment 1: What does a server look like with no agent attached?

An agent only ever sees a server through MCP. Taking the agent away shows exactly what the server offers.

The MCP Inspector is a browser page for poking at a server by hand. It has to be clicked through, so it runs outside the notebook. In PowerShell:

1. Activate the venv and move into this notebook's folder:
   - `C:\Users\kongl\Starting-something-new\venv\Scripts\Activate.ps1`
   - `cd C:\Users\kongl\Desktop\Starting-something-new\ai-agents-in-action\02_MCP_for_ai_agents`
2. Start the Inspector: `mcp dev "$PWD\mcp_research_server.py"`. The first run asks to install `@modelcontextprotocol/inspector@2.15.0`, answer `y`.
3. Open the URL it prints, e.g. `http://127.0.0.1:6274` (it may include a token, keep it)
4. Tools page: select `get_research_sources`, then **Run Tool**, and check for the three sources
5. Stop it with Ctrl+C

The cell below does the same inspection in code: list the tools, then run one directly. No LLM involved.

In [14]:
EXPERIMENTS = Path("experiments").resolve()

def stdio_server(script, name="Research Tools", **kwargs):
    """A notebook-safe stdio client for any of these servers."""
    return NotebookMCPServerStdio(
        name=name,
        params=MCPServerStdioParams(command=str(MCP_EXE), args=["run", str(script)]),
        **kwargs,
    )

async with stdio_server(RESEARCH_SCRIPT) as server:
    for tool in await server.list_tools():                     # the Inspector's Tools page
        print(f"Tool: {tool.name} - {tool.description}")
        print(f"  Input schema: {tool.input_schema}")
    result = await server.call_tool("get_research_sources", {})  # the Inspector's Run Tool button
    print("Run Tool result:", [item.text for item in result.content])

                    INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=14583037;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583038;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1084\1084]8;;\
                             204 No Content"                                                                       

                    INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=14583043;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583044;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1084\1084]8;;\
                             204 No Content"                                                                       

Tool: get_research_sources - Provides a list of research sources.
  Input schema: {'type': 'object', 'properties': {}, 'title': 'get_research_sourcesArguments'}
Run Tool result: ['Wikipedia', 'Google', 'YouTube']


## Experiment 2: If the server changes, does the agent notice?

The agent runs twice with identical code, told to "List the available research sources." The first time it talks to the original research server. The second time it talks to `research_server_two_sources.py`, where `get_research_sources` returns only two sources.

**Result:** the answer follows the server. Nothing in the agent had to change.

In [15]:
async def list_sources(server):
    """Agent logic shared by Experiments 2 and 3. It never changes."""
    agent = Agent(
        name="Assistant",
        instructions="List the available research sources.",
        mcp_servers=[server],
    )
    result = await Runner.run(agent, "Get the available research sources")
    return result.final_output

async with stdio_server(RESEARCH_SCRIPT) as server:
    print("Original server (three sources):")
    print(await list_sources(server), "\n")

async with stdio_server(EXPERIMENTS / "research_server_two_sources.py") as server:
    print("Changed server (two sources):")
    print(await list_sources(server))

Original server (three sources):


[11/11/26 23:14:34] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14583049;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583050;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

                    INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14583055;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583056;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

The available research sources are:

- Wikipedia  
- Google  
- YouTube 



Changed server (two sources):


[11/11/26 23:14:36] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14583061;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583062;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

                    INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=14583067;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583068;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1084\1084]8;;\
                             204 No Content"                                                                       

[11/11/26 23:14:37] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=14583073;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583074;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1084\1084]8;;\
                             204 No Content"                                                                       

                    INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14583079;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583080;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

The available research sources are:

- Wikipedia
- Google


## Experiment 3: Is switching transports really a one-line change?

`list_sources` from Experiment 2 is reused as is. The only change is which client gets passed in:

- **stdio:** `stdio_server(RESEARCH_SCRIPT)`, a command and a script
- **SSE:** `MCPServerSse(params={"url": "http://localhost:8000/sse"})`, just an address

The server runs with `mcp run -t sse` in the background and is always stopped at the end, even if something fails.

**Result:** same answer over SSE, same agent code.

In [16]:
def start_sse_server(command, log_file=None):
    """Start a server in the background and wait until port 8000 accepts connections."""
    process = subprocess.Popen(
        command,
        stdout=subprocess.DEVNULL,
        stderr=log_file or subprocess.DEVNULL,
    )
    for _ in range(50):
        try:
            socket.create_connection(("127.0.0.1", 8000), timeout=0.2).close()
            break
        except OSError:
            time.sleep(0.2)
    return process

sse_process = start_sse_server([str(MCP_EXE), "run", "-t", "sse", str(RESEARCH_SCRIPT)])
try:
    async with MCPServerSse(name="Research Tools", params={"url": "http://localhost:8000/sse"}) as server:  # the one-line change
        print("Over SSE:")
        print(await list_sources(server))                     # same agent logic as Experiment 2
finally:
    sse_process.terminate()                                    # always stop the server
    sse_process.wait()

[11/11/26 23:14:38] INFO     HTTP Request: GET http://localhost:8000/sse "HTTP/1.1 200 OK"          ]8;id=14583085;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583086;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\

[11/11/26 23:14:39] INFO     HTTP Request: POST                                                     ]8;id=14583091;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583092;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             http://localhost:8000/messages/?session_id=bcd87c5cf4024dcfb7124c378d7                
                             7f88f "HTTP/1.1 202 Accepted"                                                         

Over SSE:


                    INFO     HTTP Request: POST                                                     ]8;id=14583097;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583098;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             http://localhost:8000/messages/?session_id=bcd87c5cf4024dcfb7124c378d7                
                             7f88f "HTTP/1.1 202 Accepted"                                                         

[11/11/26 23:14:40] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14583103;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583104;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

                    INFO     HTTP Request: POST                                                     ]8;id=14583109;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583110;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             http://localhost:8000/messages/?session_id=bcd87c5cf4024dcfb7124c378d7                
                             7f88f "HTTP/1.1 202 Accepted"                                                         

                    INFO     HTTP Request: POST                                                     ]8;id=14583115;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583116;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             http://localhost:8000/messages/?session_id=bcd87c5cf4024dcfb7124c378d7                
                             7f88f "HTTP/1.1 202 Accepted"                                                         

                    INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14583121;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583122;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

Available research sources:
- Wikipedia
- Google
- YouTube


## Experiment 4: Can an agent be given less power?

The ready-made filesystem server from Step 5 hands an agent 16 tools, including write and move. That's far more than a file lister needs.

`mini_fs_server.py` is a small server of my own, named **Mini FS**, that sits in front of it:

- It exposes a single tool, `list_directory`
- Behind the scenes, that tool starts the real filesystem server and calls its `list_directory`
- The real server only sees the `experiments` folder

**Result:** an agent connected to Mini FS can list files and nothing else. A server can also be a client of another server, which makes wrapping a simple way to hand out less power.

In [17]:
async with stdio_server(
    EXPERIMENTS / "mini_fs_server.py",
    name="Mini FS",
    client_session_timeout_seconds=60,          # the wrapped npx server needs time to start
) as mini_fs:
    tools = await mini_fs.list_tools()
    print(f"{len(tools)} tool listed:", [tool.name for tool in tools])
    assert len(tools) == 1, "Mini FS should expose exactly one tool"

    result = await mini_fs.call_tool("list_directory", {})
    print("\nFiles in the sandbox:")
    print(result.content[0].text)

1 tool listed: ['list_directory']


[11/11/26 23:14:42] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=14583127;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583128;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1084\1084]8;;\
                             204 No Content"                                                                       


Files in the sandbox:
[FILE] mini_fs_server.py
[FILE] research_server_two_sources.py
[FILE] tracker_agent_server.log
[FILE] tracker_agent_server.py
[DIR] __pycache__


## Experiment 5: Can a whole agent become a tool?

In Step 8 the server exposed the journal's raw tools. `tracker_agent_server.py` goes one level up: it exposes a single tool, `time_tracker`, and the whole journal agent from Step 6 runs behind it.

- **Outer assistant** (this notebook) hands each request to `time_tracker`
- **Time tracker agent** (inside the server) loads the journal and records the trip

Two agents, two processes, connected by MCP. The trips below are my own.

To see both agents at work, open [OpenAI Traces](https://platform.openai.com/traces) and look for:

- `journal_assistant (outer assistant)`
- `time_tracker (MCP-hosted agent)`, one per request it handled

In [18]:
from agents import flush_traces, trace

my_trips = [
    "Watched the Great Pyramid of Giza being built",
    "Sat in the audience for the first performance of Hamlet in 1600",
    "Saw the Wright brothers' first flight in 1903",
]

tracker_agent_log = open(EXPERIMENTS / "tracker_agent_server.log", "w")
tracker_agent_process = start_sse_server(
    [sys.executable, str(EXPERIMENTS / "tracker_agent_server.py")], tracker_agent_log
)
try:
    async with MCPServerSse(
        name="Time Tracker Agent",
        params={"url": "http://localhost:8000/sse"},
        client_session_timeout_seconds=60,        # each call runs a whole agent, so allow time
    ) as tracker_server:
        assistant = Agent(
            name="Assistant",
            instructions=(
                "You help a time traveler keep a journal. "
                "Hand every request to the time_tracker tool: send each new trip to record it, "
                "and ask it for the full history when a summary is wanted."
            ),
            mcp_servers=[tracker_server],
        )
        with trace("journal_assistant (outer assistant)"):
            for trip in my_trips:
                await Runner.run(assistant, trip)
            summary = await Runner.run(assistant, "Show my travel history")
        flush_traces()

    print("Summary of all recorded events:")
    print(summary.final_output)
finally:
    tracker_agent_process.terminate()
    tracker_agent_process.wait()
    tracker_agent_log.close()

print("\nInside the server, the tracker agent did this:")
with open(EXPERIMENTS / "tracker_agent_server.log") as f:
    for line in f:
        if "Event recorded" in line or "Loading" in line:
            print(line.replace("INFO", "").strip())   # drop the logger's prefix when present

[11/11/26 23:14:45] INFO     HTTP Request: GET http://localhost:8000/sse "HTTP/1.1 200 OK"          ]8;id=14583133;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583134;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\

                    INFO     HTTP Request: POST                                                     ]8;id=14583139;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583140;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             http://localhost:8000/messages/?session_id=021a01e9b6e74a64879ea4fdee4                
                             e9f69 "HTTP/1.1 202 Accepted"                                                         

                    INFO     HTTP Request: POST                                                     ]8;id=14583145;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583146;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             http://localhost:8000/messages/?session_id=021a01e9b6e74a64879ea4fdee4                
                             e9f69 "HTTP/1.1 202 Accepted"                                                         

[11/11/26 23:14:46] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14583151;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583152;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

                    INFO     HTTP Request: POST                                                     ]8;id=14583157;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583158;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             http://localhost:8000/messages/?session_id=021a01e9b6e74a64879ea4fdee4                
                             e9f69 "HTTP/1.1 202 Accepted"                                                         

[11/11/26 23:14:48] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=14583163;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583164;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1084\1084]8;;\
                             204 No Content"                                                                       

... (20 outputs left out)

[11/11/26 23:15:09] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=14583289;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583290;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1084\1084]8;;\
                             204 No Content"                                                                       

                    INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=14583295;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583296;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             OK"                                                                                   

                    INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=14583301;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=14583302;file://C:\Users\kongl\Starting-something-new\venv\Lib\site-packages\httpx2\_client.py#1084\1084]8;;\
                             204 No Content"                                                                       

Summary of all recorded events:
Here’s your full time travel history so far:

1. Watched the Great Pyramid of Giza being built  
2. Sat in the audience for the first performance of Hamlet in 1600  
3. Saw the Wright brothers' first flight in 1903

Inside the server, the tracker agent did this:
Loading journal entries...
Event recorded: Watched the Great Pyramid of Giza being built
Loading journal entries...
Event recorded: Sat in the audience for the first performance of Hamlet in 1600
Loading journal entries...
Event recorded: Saw the Wright brothers' first flight in 1903.
Loading journal entries...
